In [1]:
# 필요한 라이브러리 불러오기.
import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from datetime import datetime
from itertools import product
import wandb

# 데이터 import를 위한 path 추가.
import os
import sys
if os.name == 'nt':
    BASE_PATH = 'C:\\Users\\first\\Develop\\Kut-Deep-Learning-250204\\_03_homeworks\\homework_2\\'
else:
    BASE_PATH = '/home/ksy/Develop/Kut-Deep-Learning-250204/_03_homeworks/homework_2/'
sys.path.insert(1, BASE_PATH)

# 타이타닉 데이터 모듈 불러오기.
import titanic_dataset as tdata

# 넘파이 관련 경고 제거. (3.0에서 달라지는 API 경고. 사용 중인 넘파이 버전은 2.x임.)
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [2]:
def get_data():
    train_dataset, validation_dataset, test_dataset = tdata.get_preprocessed_dataset()

    train_data_loader = DataLoader(dataset=train_dataset, batch_size=wandb.config.batch_size, shuffle=True)
    validation_data_loader = DataLoader(dataset=validation_dataset, batch_size=len(validation_dataset))
    test_data_loader = DataLoader(dataset=test_dataset, batch_size=len(test_dataset))

    return train_data_loader, validation_data_loader, test_data_loader

In [3]:
class MyModel(nn.Module):
    def __init__(self, n_input_data, n_output, act=nn.ReLU):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(n_input_data, wandb.config.n_hidden_unit_list[0]),
            act(),
            nn.Linear(wandb.config.n_hidden_unit_list[0], wandb.config.n_hidden_unit_list[1]),
            act(),
            nn.Linear(wandb.config.n_hidden_unit_list[1], n_output),
        )

    def forward(self, x):
        x = self.model(x)
        return x

In [4]:
def training_loop(model, optimizer, train_data_loader, validation_data_loader):
    n_epochs = wandb.config.epochs
    loss_fn = nn.MSELoss()  # Use a built-in loss function
    next_print_epoch = 100

    for epoch in range(1, n_epochs + 1):
        loss_train = 0.0
        num_trains = 0
        for train_batch in train_data_loader:
            input_data, target = train_batch
            output_train = model(input_data)
            loss = loss_fn(output_train, target)
            loss_train += loss.item()
            num_trains += 1

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        loss_validation = 0.0
        num_validations = 0
        with torch.no_grad():
            for validation_batch in validation_data_loader:
                input_data, target = validation_batch
                output_validation = model(input_data)
                loss = loss_fn(output_validation, target)
                loss_validation += loss.item()
                num_validations += 1

        wandb.log({
            "Epoch": epoch,
            "Training loss": loss_train / num_trains,
            "Validation loss": loss_validation / num_validations
        })

        if epoch >= next_print_epoch:
            print(
                f"Epoch {epoch}, "
                f"Training loss {loss_train / num_trains:.4f}, "
                f"Validation loss {loss_validation / num_validations:.4f}"
            )
            next_print_epoch += 100

In [5]:
def main(args):
    current_time_str = datetime.now().astimezone().strftime('%Y-%m-%d_%H-%M-%S')

    config = {
        'epochs': args.epochs,
        'batch_size': args.batch_size,
        'learning_rate': 1e-3,
        'n_hidden_unit_list': [20, 20],
    }

    wandb.init(
        mode="online" if args.wandb else "disabled",
        project="titanic",
        notes="Titanic Competition",
        tags=["titanic"],
        name=current_time_str,
        config=config
    )
    
    if args.log_level >= 2:
        print(args)
        print(wandb.config)
        
    train_data_loader, validation_data_loader, test_data_loader = get_data()
    
    def get_model_and_optimizer():
        my_model = MyModel(n_input_data=10, n_output=1, act=args.act)
        optimizer = optim.SGD(my_model.parameters(), lr=wandb.config.learning_rate)

        return my_model, optimizer

    linear_model, optimizer = get_model_and_optimizer()

    if args.log_level >= 2:
        print("#" * 50, 1)

    if args.log_level >= 1:
        print(f'[INFO] :: Begin Training... ::')
        print(f'[INFO] {args}')

    training_loop(
        model=linear_model,
        optimizer=optimizer,
        train_data_loader=train_data_loader,
        validation_data_loader=validation_data_loader
    )
    wandb.finish()

    if args.log_level >= 1:
        print(f'[INFO] :: Training Done! ::')

    if args.valtest:
        test_data, target_data = next(iter(validation_data_loader))
        res = linear_model.forward(test_data)
        res = torch.round(torch.clamp(res, min=0, max=1))

        total_count = len(target_data)
        correct_count = int(torch.count_nonzero(res == target_data))
        
        if args.log_level >= 1:
            print(f'[INFO] :: Testing on Validation Dataset ::')
            print(f'[INFO] {correct_count}/{total_count}')
            print(f'[INFO] {correct_count / total_count * 100:.2f}')
    else:
        test_data = next(iter(test_data_loader))
        res = torch.round(torch.clamp(linear_model.forward(test_data), min=0))

        with open(f'{BASE_PATH}{args.out}', 'w') as f:
            f.write('PassengerId,Survived\n')
            for i, v in zip(range(892, 1309 + 1), res):
                f.write(f'{i},{int(v)}\n')
                
            if args.log_level >= 1:
                print(f'[INFO] `{f.name}` saved')

In [6]:
# stolen from: https://arxiv.org/pdf/2302.13696
class MoLU(nn.Module):
    __constants__ = ["inplace"]
    inplace: bool

    def __init__(self, inplace: bool = False) -> None:
        super().__init__()
        self.inplace = inplace

    def forward(self, input: torch.Tensor) -> torch.Tensor:
        """
        Runs the forward pass.
        """
        x = input.clone().detach()
        return input.mul_(torch.mul(torch.add(torch.tanh(x), 1), 0.5))

    def extra_repr(self) -> str:
        """
        Return the extra representation of the module.
        """
        inplace_str = "inplace=True" if self.inplace else ""
        return inplace_str

In [8]:
# https://docs.wandb.ai/guides/track/config

# stolen from: https://stackoverflow.com/questions/2352181/how-to-use-a-dot-to-access-members-of-dictionary
class dotdict(dict):
    """dot.notation access to dictionary attributes"""
    __getattr__ = dict.get
    __setattr__ = dict.__setitem__
    __delattr__ = dict.__delitem__

act = [nn.Sigmoid, nn.ReLU, nn.ELU, nn.LeakyReLU, MoLU]
batch = [16, 32, 64, 128]

# gen csv
for a, b in product(act, batch):
    args = dotdict({
        'wandb': True,
        'act': a,
        'batch_size': b,
        'epochs': 1000,
        'valtest': False,
        'out': f'{a.__name__}_{b}.csv',
        'log_level': 1,
    })
    main(args)

# val test
for a, b in product(act, batch):
    args = dotdict({
        'wandb': False,
        'act': a,
        'batch_size': b,
        'epochs': 1000,
        'valtest': True,
        'out': 'out.csv',
        'log_level': 1,
    })
    main(args)

[INFO] :: Begin Training... ::
[INFO] {'wandb': True, 'act': <class 'torch.nn.modules.activation.Sigmoid'>, 'batch_size': 16, 'epochs': 1000, 'valtest': False, 'out': 'Sigmoid_16.csv', 'log_level': 1}
Epoch 100, Training loss 0.2147, Validation loss 0.2247
Epoch 200, Training loss 0.2068, Validation loss 0.2211
Epoch 300, Training loss 0.2028, Validation loss 0.2211
Epoch 400, Training loss 0.2014, Validation loss 0.2211
Epoch 500, Training loss 0.2010, Validation loss 0.2206
Epoch 600, Training loss 0.2013, Validation loss 0.2205
Epoch 700, Training loss 0.2007, Validation loss 0.2202
Epoch 800, Training loss 0.2000, Validation loss 0.2204
Epoch 900, Training loss 0.1988, Validation loss 0.2195
Epoch 1000, Training loss 0.1986, Validation loss 0.2194


Epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▅▅▅▅▆▆▇▇▇▇▇▇▇▇▇█████
Training loss,█▄▃▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation loss,██▆▅▄▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁
Epoch,1000
Training loss,0.19856
Validation loss,0.21943


[INFO] :: Training Done! ::
[INFO] `C:\Users\first\Develop\Kut-Deep-Learning-250204\_03_homeworks\homework_2\Sigmoid_16.csv` saved


[INFO] :: Begin Training... ::
[INFO] {'wandb': True, 'act': <class 'torch.nn.modules.activation.Sigmoid'>, 'batch_size': 32, 'epochs': 1000, 'valtest': False, 'out': 'Sigmoid_32.csv', 'log_level': 1}
Epoch 100, Training loss 0.2396, Validation loss 0.2306
Epoch 200, Training loss 0.2325, Validation loss 0.2224
Epoch 300, Training loss 0.2293, Validation loss 0.2157
Epoch 400, Training loss 0.2201, Validation loss 0.2063
Epoch 500, Training loss 0.2157, Validation loss 0.1994
Epoch 600, Training loss 0.2159, Validation loss 0.1951
Epoch 700, Training loss 0.2138, Validation loss 0.1924
Epoch 800, Training loss 0.2118, Validation loss 0.1898
Epoch 900, Training loss 0.2103, Validation loss 0.1887
Epoch 1000, Training loss 0.2109, Validation loss 0.1880


Epoch,▁▁▁▁▁▂▂▂▃▃▄▄▄▄▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
Training loss,████▇▆▆▆▅▅▄▄▃▄▄▃▃▂▃▂▂▂▂▂▂▁▂▂▁▁▁▁▁▁▁▁▁▁▁▁
Validation loss,█▇▇▇▆▆▅▅▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁
Epoch,1000
Training loss,0.2109
Validation loss,0.18798


...

Epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇▇██
Training loss,█▆▄▄▄▄▄▃▃▃▃▃▃▃▃▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▂▂
Validation loss,▆▅▃█▃▂▂▃▂▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▁▁▄▁▁▃▁▁▂▄▁▁▂
Epoch,1000
Training loss,0.17211
Validation loss,0.19111


[INFO] :: Training Done! ::
[INFO] `C:\Users\first\Develop\Kut-Deep-Learning-250204\_03_homeworks\homework_2\MoLU_64.csv` saved


[INFO] :: Begin Training... ::
[INFO] {'wandb': True, 'act': <class '__main__.MoLU'>, 'batch_size': 128, 'epochs': 1000, 'valtest': False, 'out': 'MoLU_128.csv', 'log_level': 1}
Epoch 100, Training loss 0.2244, Validation loss 0.2314
Epoch 200, Training loss 0.2115, Validation loss 0.2253
Epoch 300, Training loss 0.2014, Validation loss 0.2324
Epoch 400, Training loss 0.1968, Validation loss 0.2143
Epoch 500, Training loss 0.1907, Validation loss 0.2253
Epoch 600, Training loss 0.1838, Validation loss 0.2032
Epoch 700, Training loss 0.1826, Validation loss 0.2029
Epoch 800, Training loss 0.1764, Validation loss 0.1966
Epoch 900, Training loss 0.1771, Validation loss 0.1993
Epoch 1000, Training loss 0.1744, Validation loss 0.1890


Epoch,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇█████
Training loss,█▇▆▅▅▅▄▄▄▄▃▃▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▂▁▂▁▂▁▁▁
Validation loss,█▆▄▄▄▄▄▄▃▃▃▃▃▆▃▃▃▃▃▃▂▂▂▂▂▂▂▅▂▃▂▁▂▂▂▁▁▁▂▁
Epoch,1000
Training loss,0.17443
Validation loss,0.18898


[INFO] :: Training Done! ::
[INFO] `C:\Users\first\Develop\Kut-Deep-Learning-250204\_03_homeworks\homework_2\MoLU_128.csv` saved
[INFO] :: Begin Training... ::
[INFO] {'wandb': False, 'act': <class 'torch.nn.modules.activation.Sigmoid'>, 'batch_size': 16, 'epochs': 1000, 'valtest': True, 'out': 'out.csv', 'log_level': 1}
Epoch 100, Training loss 0.2212, Validation loss 0.2293
Epoch 200, Training loss 0.2089, Validation loss 0.2267
Epoch 300, Training loss 0.2026, Validation loss 0.2281
Epoch 400, Training loss 0.1994, Validation loss 0.2299
Epoch 500, Training loss 0.1978, Validation loss 0.2287
Epoch 600, Training loss 0.1972, Validation loss 0.2288
Epoch 700, Training loss 0.1953, Validation loss 0.2275
Epoch 800, Training loss 0.1943, Validation loss 0.2248
Epoch 900, Training loss 0.1928, Validation loss 0.2244
Epoch 1000, Training loss 0.1917, Validation loss 0.2240
[INFO] :: Training Done! ::
[INFO] :: Testing on Validation Dataset ::
[INFO] 112/178
[INFO] 62.92
[INFO] :: Begin T